# 03b2 · DMID mass crops — ROI masks only (fixed + faster)

**Runtime: CPU.** Replaces 03b. Finding from 03b: the `Pixel-level annotation` masks outline the **whole breast**
(31 % of the image, crops showed the full breast), so they cannot be a lesion test set. The `ROI Masks` are the lesion masks
(median 0.6 % of the image, usually one component). 03b also chose the y-convention with the breast masks, which was not informative.

This notebook: ROI masks only · y-convention tested on ROI masks by hit rate · each TIFF read once (float32) · crops written to
local disk first and copied to Drive at the end (Drive writes were slowing 03b to minutes per crop).
Output: `data/external/dmid_mass_256/roi/images|masks + metadata.csv + qc_roi.png`. The old `pixel/` folder is left untouched.


In [ ]:
# 1 · Drive, packages, download (local disk)
from google.colab import drive
drive.mount('/content/drive')
import os, re, glob, json, subprocess, requests, numpy as np, pandas as pd, cv2
subprocess.run(['pip', '-q', 'install', 'tifffile', 'imagecodecs'], check=False)
import tifffile
from PIL import Image
EXT = '/content/drive/MyDrive/quantum-placement/data/external'
OUT = f'{EXT}/dmid_mass_256'; os.makedirs(OUT, exist_ok=True)
RAW, LOCAL = '/content/dmid_raw', '/content/dmid'
os.makedirs(RAW, exist_ok=True); os.makedirs(LOCAL, exist_ok=True)
art = requests.get('https://api.figshare.com/v2/articles/24522883', timeout=60).json()
for f in art['files']:
    if not any(k in f['name'] for k in ('TIFF', 'Pixel', 'ROI', 'Metadata')):
        continue
    dst = f"{RAW}/{f['name']}"
    if not os.path.exists(dst) or os.path.getsize(dst) != f['size']:
        with requests.get(f['download_url'], stream=True, timeout=900) as r:
            r.raise_for_status()
            with open(dst, 'wb') as o:
                for ch in r.iter_content(1 << 20): o.write(ch)
    if dst.endswith('.zip') and not os.path.isdir(f"{LOCAL}/{f['name'][:-4]}"):
        subprocess.run(['unzip', '-q', '-o', dst, '-d', LOCAL], check=False)
    print('ok', f['name'])
print(sorted(os.listdir(LOCAL)))


In [ ]:
# 2 · Helpers (same windowing and crop as notebook 01 / CBIS)
def read_tif(p):
    try:
        a = tifffile.imread(p)
    except Exception:
        a = np.array(Image.open(p))
    a = np.asarray(a)
    if a.ndim == 3 and a.shape[0] in (1, 3, 4) and a.shape[-1] not in (1, 3, 4):
        a = np.moveaxis(a, 0, -1)
    if a.ndim == 3:
        a = a[..., :3].mean(-1) if a.shape[-1] >= 3 else a[..., 0]
    return a

def to_uint8(img):
    img = img.astype(np.float32); nz = img[img > 0]
    lo, hi = (np.percentile(nz, 1), np.percentile(nz, 99.5)) if nz.size else (img.min(), img.max())
    return (np.clip((img - lo) / max(hi - lo, 1e-6), 0, 1) * 255).astype(np.uint8)

def roi_crop(img, mask, size=256, margin=0.2):
    ys, xs = np.where(mask > 0)
    if ys.size == 0: return None, None, None
    y0, y1, x0, x1 = ys.min(), ys.max() + 1, xs.min(), xs.max() + 1
    side = max(int(max(y1 - y0, x1 - x0) * (1 + 2 * margin)), 64)
    cy, cx = (y0 + y1) // 2, (x0 + x1) // 2
    H, W = mask.shape; t, l = cy - side // 2, cx - side // 2
    pad = max(0, -t, -l, t + side - H, l + side - W)
    if pad:
        img = np.pad(img, pad); mask = np.pad(mask, pad); t += pad; l += pad
    ic = cv2.resize(img[t:t + side, l:l + side], (size, size), interpolation=cv2.INTER_AREA)
    mc = cv2.resize(mask[t:t + side, l:l + side].astype(np.uint8), (size, size), interpolation=cv2.INTER_NEAREST)
    return ic, (mc > 0).astype(np.uint8) * 255, dict(bbox_h=int(y1 - y0), bbox_w=int(x1 - x0), crop_side=side, padded=bool(pad))

info_path = glob.glob(f'{LOCAL}/**/Info.txt', recursive=True)[0]
def _f(s):  # Info.txt has non-numeric entries such as '-' -> NaN
    try: return float(s)
    except ValueError: return np.nan
rows = []
for line in open(info_path, errors='ignore'):
    p = line.split()
    if len(p) >= 3 and p[0].upper().startswith('IMG'):
        rows.append(dict(img=p[0].upper(), tissue=p[1], cls=p[2], sev=p[3] if len(p) > 3 else None,
                         x=_f(p[4]) if len(p) > 6 else np.nan, y=_f(p[5]) if len(p) > 6 else np.nan,
                         r=_f(p[6]) if len(p) > 6 else np.nan))
info = pd.DataFrame(rows)
print('Info rows:', len(info), '| images:', info.img.nunique()); print(info.cls.value_counts().to_string())
print(pd.crosstab(info.cls, info.sev).to_string())
meta = pd.read_excel(glob.glob(f'{RAW}/*Metadata*')[0]); print('\nMetadata', meta.shape); print(meta.head(8).to_string())


In [ ]:
# 3 · y-convention of Info.txt, tested on the ROI (lesion) masks
import gc, shutil, time
IMG_DIR = glob.glob(f'{LOCAL}/**/TIFF Images', recursive=True)[0]
ROI_DIR = glob.glob(f'{LOCAL}/**/ROI Masks', recursive=True)[0]
mass = info[info.cls.isin(['CIRC', 'SPIC', 'MISC']) & info.x.notna() & info.y.notna()].copy()
mass['r'] = mass.r.fillna(50).clip(lower=20)
print('mass rows:', len(mass), '| images:', mass.img.nunique(),
      '| with ROI mask file:', int(mass.img.map(lambda i: os.path.exists(f'{ROI_DIR}/{i}.tif')).sum()))

def comps(m):
    n, lab, st, cen = cv2.connectedComponentsWithStats(m.astype(np.uint8), connectivity=8)
    return n, lab, st, cen

def pick(n, lab, st, cen, x, y, r):
    # components whose bbox comes within 1.5 r of the listed centre
    keep = []
    for k in range(1, n):
        x0, y0, w, h, _ = st[k]
        dx = max(x0 - x, 0, x - (x0 + w - 1)); dy = max(y0 - y, 0, y - (y0 + h - 1))
        if dx * dx + dy * dy <= (1.5 * r) ** 2: keep.append(k)
    return keep

hit = {'top': 0, 'bottom': 0}; tested = 0
for _, r in mass.drop_duplicates('img').head(60).iterrows():
    f = f'{ROI_DIR}/{r.img}.tif'
    if not os.path.exists(f): continue
    m = read_tif(f) > 0; H = m.shape[0]; c = comps(m); tested += 1
    hit['top'] += bool(pick(*c, r.x, r.y, r.r)); hit['bottom'] += bool(pick(*c, r.x, H - r.y, r.r))
print(f'hit rate on {tested} images:', {k: round(v / max(tested, 1), 3) for k, v in hit.items()})
FLIP_Y = hit['bottom'] > hit['top']
print('Using y measured from the', 'BOTTOM' if FLIP_Y else 'TOP')


In [ ]:
# 4 · Build ROI crops (one per listed mass; only that mass's mask component(s) kept). Local first, then Drive.
from tqdm.auto import tqdm
TMP = '/content/dmid_roi_256'
os.makedirs(f'{TMP}/images', exist_ok=True); os.makedirs(f'{TMP}/masks', exist_ok=True)
rec, miss = [], []
t0 = time.time()
mass = mass.reset_index(drop=True)
for img_id, grp in tqdm(list(mass.groupby('img', sort=True)), desc='images'):
    f = f'{ROI_DIR}/{img_id}.tif'
    if not os.path.exists(f):
        miss += [(img_id, 'no ROI mask file')] * len(grp); continue
    m = read_tif(f) > 0
    im = read_tif(f'{IMG_DIR}/{img_id}.tif').astype(np.float32)
    if im.shape != m.shape:
        m = cv2.resize(m.astype(np.uint8), (im.shape[1], im.shape[0]), interpolation=cv2.INTER_NEAREST) > 0
    im8 = to_uint8(im); del im
    c = comps(m)
    for i, r in grp.iterrows():
        y = m.shape[0] - r.y if FLIP_Y else r.y
        keep = pick(*c, r.x, y, r.r)
        if not keep: miss.append((img_id, 'no mask component at listed position')); continue
        mm = np.isin(c[1], keep)
        ic, mc, meta_c = roi_crop(im8, mm)
        rid = f'{img_id}_{i}'
        cv2.imwrite(f'{TMP}/images/{rid}.png', ic); cv2.imwrite(f'{TMP}/masks/{rid}.png', mc)
        rec.append(dict(roi_id=rid, img=img_id, patient=img_id, cls=r.cls, sev=r.sev, tissue=r.tissue,
                        n_comp=len(keep), mask_frac=round(float((mc > 0).mean()), 4), **meta_c))
    del m, im8, c; gc.collect()
md_ = pd.DataFrame(rec); md_.to_csv(f'{TMP}/metadata.csv', index=False)
print(f'roi: {len(md_)} mass crops from {md_.img.nunique()} images in {time.time() - t0:.0f}s | skipped {len(miss)}')
print(pd.Series([w for _, w in miss]).value_counts().to_string() if miss else '')
dst = f'{OUT}/roi'
if os.path.exists(dst): print('NOTE: roi/ already exists on Drive, files are overwritten')
shutil.copytree(TMP, dst, dirs_exist_ok=True)
print('copied to', dst)


In [ ]:
# 5 · QC: 12 crops with the mask outline + summary
import matplotlib.pyplot as plt
md_ = pd.read_csv(f'{OUT}/roi/metadata.csv')
pick12 = md_.sample(min(12, len(md_)), random_state=0).roi_id.tolist()
fig, ax = plt.subplots(2, 6, figsize=(15, 5.4))
for a, rid in zip(ax.ravel(), pick12):
    im = cv2.imread(f'{OUT}/roi/images/{rid}.png', 0); mk = cv2.imread(f'{OUT}/roi/masks/{rid}.png', 0)
    rgb = cv2.cvtColor(im, cv2.COLOR_GRAY2RGB)
    cs, _ = cv2.findContours(mk, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE); cv2.drawContours(rgb, cs, -1, (255, 60, 60), 1)
    a.imshow(rgb); a.set_title(rid, fontsize=7); a.axis('off')
plt.tight_layout(); plt.savefig(f'{OUT}/qc_roi.png', dpi=110); plt.show()
print('crops', len(md_), '| by class', md_.cls.value_counts().to_dict(), '| severity', md_.sev.value_counts().to_dict(),
      '| mask_frac median', round(md_.mask_frac.median(), 3), '| padded', int(md_.padded.sum()),
      '| >1 component', int((md_.n_comp > 1).sum()))
